In [0]:

-- 1. 删除旧表
DROP TABLE IF EXISTS adhyivy.default.gold_dim_sku;

-- 2. 创建新表
CREATE TABLE adhyivy.default.gold_dim_sku 
USING DELTA
LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/gold/dim_sku'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
with
sku as
(
    select
        id,
        price,
        sku_name,
        sku_desc,
        weight,
        is_sale,
        spu_id,
        category3_id,
        tm_id,
        create_time
    from silver_sku_info
),
spu as
(
    select
        id,
        spu_name
    from silver_spu_info
),
c3 as
(
    select
        id,
        name,
        category2_id
    from silver_base_category3
),
c2 as
(
    select
        id,
        name,
        category1_id
    from silver_base_category2
),
c1 as
(
    select
        id,
        name
    from silver_base_category1
),
tm as
(
    select
        id,
        tm_name
    from silver_base_trademark
)
select
    sku.id,
    sku.price,
    sku.sku_name,
    sku.sku_desc,
    sku.weight,
    sku.is_sale,
    sku.spu_id,
    spu.spu_name,
    sku.category3_id,
    c3.name,
    c3.category2_id,
    c2.name,
    c2.category1_id,
    c1.name,
    sku.tm_id,
    tm.tm_name,
    CURRENT_TIMESTAMP() as load_time
from sku
left join spu on sku.spu_id=spu.id
left join c3 on sku.category3_id=c3.id
left join c2 on c3.category2_id=c2.id
left join c1 on c2.category1_id=c1.id
left join tm on sku.tm_id=tm.id
;